# Spherical Fourier-Bessel $C_\ell(k)$ for the Ly-$\alpha$ Forest: Periodic Box

**Pipeline:** 3D GRF → extract sightlines → LOS DFT → SHT → pseudo-$C_\ell(k)$ → theory comparison

**Key facts for periodic box with uniform weights:**
- Data weights: $w_j = \text{FT}[\delta_F]_j(k)$ — the LOS DFT of flux fluctuations
- Mask weights: $m_j = \text{FT}[1]_j(k=0) = N$ (sum of ones), zero at $k \neq 0$
- pseudo-$C_\ell = |a_{\ell m}^{\rm data}|^2 / (2\ell+1)$ — **no D−R subtraction** for Ly-$\alpha$
- Theory: $C_\ell^{\rm true}(k) = P_F(\ell/\bar\chi, k) / (32\pi^3 \bar\chi^2)$ for use with `MaskDeconvolution`
- **No shot noise subtraction:** $\sum_j w_j^2 / (4\pi)$ is cosmological signal (the $j{=}k$ pair-counting diagonal), not Poisson noise. The MASTER framework $\langle\hat{C}_\ell\rangle = M_{\ell L}\,C_L^{\rm true}$ already accounts for this through the full window function.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import sys, os, time, gc

# Add project paths
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.getcwd())

from sht.sht import DirectSHT
from sht.mask_deconvolution import MaskDeconvolution
from sht.theory_lya import (theory_cl_for_deconvolution,
                             compute_chi_bar_from_grid)
import GRF_class as my_GRF
import SHT_lya as sht_lya
import fast_Wigner3j as Wigner3j

%matplotlib inline
plt.rcParams.update({'font.size': 13, 'figure.figsize': (10, 6)})

### Parameters

In [ ]:
# GRF / survey settings
add_rsd_  = False       # set True to include Kaiser RSD
num_qso   = 9797        # requested number of sightlines
chi_shift = 5000.0      # LOS displacement (Mpc/h)

# SHT settings
Nl         = 500
lambda_max = 1000
Nx         = 2 * Nl
xmax       = 3. / 4.
NperBin    = 32          # bandpower bin width

# k=0 is the only mode with non-zero mask FT for periodic box
k_idx = 0

# Number of independent GRF realisations (increase for smaller error bars)
num_sim = 20

print(f'Nl={Nl}, lambda_max={lambda_max}, num_qso={num_qso}')
print(f'num_sim={num_sim}, k_idx={k_idx}, add_rsd={add_rsd_}')

## 1. Simulation Loop — measured pseudo-$C_\ell(k{=}0)$

For each seed: generate 3D GRF, extract sightlines, LOS DFT, SHT at $k{=}0$.

Accumulate `hp.alm2cl(hdat)` across realisations.

In [ ]:
sht_eng = DirectSHT(Nl, Nx, xmax)
print(f'DirectSHT: Nl={Nl}, Nx={Nx}, xmax={xmax}')

In [ ]:
cl_stack = []  # measured pseudo-Cl(k=0)
wl_ref   = None

for sim_idx in range(num_sim):
    seed = 1000 + sim_idx
    t0 = time.time()

    GRF = my_GRF.PowerSpectrumGenerator(add_rsd=add_rsd_, seed=seed)
    all_x, all_y, all_z, all_w_rand, all_w_gal, Nskew = GRF.process_skewers(
        Nskew=num_qso, shift=chi_shift)
    all_theta, all_phi = GRF.compute_theta_phi_skewer_start(
        all_x[:, 0], all_y[:, 0], all_z[:, 0])
    chi_grid = all_x[0, :]
    delta_F  = all_w_gal - 1.0

    # LOS DFT — unnormalised, real-only output (original convention)
    k_arr, FT_mask, FT_delta = sht_lya.compute_dft(chi_grid, all_w_rand, delta_F)
    N = chi_grid.size

    # Periodic-box sanity check
    if sim_idx == 0:
        assert np.allclose(FT_mask[:, 0], N), f'FT_mask[:,0] != N'
        assert np.allclose(FT_mask[:, 1], 0, atol=1e-10), 'FT_mask[:,1] != 0'
        print(f'  ✓ FT_mask[:,0]={N}, FT_mask[:,k≠0]=0  (periodic box)')

    # SHT + pseudo-Cl  (no SN subtraction — the diagonal Σw²/4π is cosmological
    # signal, not Poisson noise; see test_fullsky_v2.py for derivation)
    w_data = FT_delta[:, k_idx]
    hdat = sht_eng(all_theta, all_phi, w_data)
    cl   = hp.alm2cl(hdat)[:Nl]
    cl_stack.append(cl)

    # Save geometry from first sim (process_skewers uses a fixed random seed
    # for sightline selection, so angular positions are identical across sims)
    if sim_idx == 0:
        hran       = sht_eng(all_theta, all_phi, FT_mask[:, k_idx])
        wl_ref     = hp.alm2cl(hran)[:Nl]
        chi_grid_0 = chi_grid
        theta_0, phi_0 = all_theta, all_phi
        Nskew_0    = Nskew
        plin_ref   = GRF.plin
        b1_ref     = GRF.my_bias

    del GRF, all_x, all_y, all_z, all_w_rand, all_w_gal, delta_F, FT_mask, FT_delta
    gc.collect()
    print(f'  sim {sim_idx}: seed={seed}, Nskew={Nskew}, dt={time.time()-t0:.1f}s')

cl_stack = np.array(cl_stack)
cl_mean  = np.mean(cl_stack, axis=0)
cl_std   = np.std(cl_stack, axis=0)

dchi    = chi_grid_0[1] - chi_grid_0[0]
L_box   = N * dchi
chi_bar = compute_chi_bar_from_grid(chi_grid_0)
print(f'\nNskew={Nskew_0}, N={N}, L_box={L_box:.1f}, dchi={dchi:.4f}, chi_bar={chi_bar:.1f}')
print(f'b1={b1_ref}')

### Diagnostic: sightline sky positions & example LOS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(np.degrees(phi_0), np.degrees(theta_0), s=0.3, alpha=0.3)
ax.set_xlabel(r'$\phi$ (deg)')
ax.set_ylabel(r'$\theta$ (deg)')
ax.set_title(f'{Nskew_0} sightlines on the sky')

ax = axes[1]
ells = np.arange(Nl)
for i in range(min(num_sim, 3)):
    ax.semilogy(ells[2:], cl_stack[i, 2:], alpha=0.4, label=f'sim {i}')
ax.semilogy(ells[2:], cl_mean[2:], 'k-', lw=2, label='mean')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'pseudo-$C_\ell(k{=}0)$')
ax.set_title('Measured pseudo-$C_\ell$ across realisations')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 2. Pair-Counting Theory (original code approach)

The original pipeline theory prediction is:

$$C_{\rm theory} = \frac{\mathbf{M}_{pk} \cdot \text{PLKjKk}}{4\pi \cdot 2\pi\bar\chi^2},$$

plotted as $C_{\rm theory}/(4\pi)^2$, where:
- $\mathbf{M}_{pk}[\ell, \lambda] = \frac{2\lambda+1}{4\pi}\sum_L (2L+1)\, P_F(L/\bar\chi)\,(3j)^2$ — Wigner coupling with $P_F$ as weights
- $\text{PLKjKk}[\lambda] = K_jK_k \sum_{j,k} P_\lambda(\cos\theta_{jk})$ — pair-counting angular window

### Why the factor $1/(32\pi^3)$?

Using two verified algebraic identities:

1. $\text{PLKjKk}[\lambda] = 4\pi \, W_\lambda$  (pair count = $4\pi$ × angular $C_\ell$ of mask)
2. $\mathbf{M}_{pk} \cdot W = \mathbf{M}_{W} \cdot P$  (Wigner 3j symmetry: spectrum and window are exchangeable)

we get

$$\frac{C_{\rm theory}}{(4\pi)^2} = \frac{\mathbf{M}_W \cdot P}{2\pi\bar\chi^2 \cdot (4\pi)^2} = \sum_L M_{\ell L} \frac{P_L}{32\pi^3 \bar\chi^2}$$

Comparing to the MASTER relation $\langle\tilde{C}_\ell\rangle = \sum_L M_{\ell L} C_L^{\rm true}$ identifies:

$$C_L^{\rm true} = \frac{P_F(L/\bar\chi, k)}{32\pi^3 \bar\chi^2} = \frac{P_F}{\underbrace{2\pi}_{\text{sFB projection}} \cdot \underbrace{4\pi}_{\text{addition thm}} \cdot \underbrace{(4\pi)}_{\text{coupling norm}} \cdot \bar\chi^2}$$

Each factor has a standard origin in the spherical harmonic normalization conventions.

In [ ]:
# Pair counting: PLKjKk[λ] = KjKk × Σ_{j,k} Pλ(cos θ_{jk})
nhat = sht_lya.compute_nhat(theta_0, phi_0)
cos_theta = np.dot(nhat, nhat.T)
KjKk = N**2  # periodic box: FT_mask[:,0] = N
del nhat; gc.collect()

t0 = time.time()
print('Computing pair-counting Legendre sums...', end='', flush=True)
PLKjKk = sht_lya.legendre_polynomials_sum(lambda_max, cos_theta, KjKk)[:lambda_max]
print(f'done ({time.time()-t0:.1f}s)')
del cos_theta; gc.collect()

# Verify identity: PLKjKk[λ] = 4π × wl_ref[λ]
ratio_check = PLKjKk[:10] / (4 * np.pi * wl_ref[:10])
print(f'✓ PLKjKk / (4π × wl_ref) = {ratio_check[:5].round(6)} (should be 1.0)')

In [ ]:
# P(L/chi_bar) with bias factor
L_range = np.arange(lambda_max, dtype=float)
pk_L = b1_ref**2 * plin_ref(L_range / chi_bar)

# Wigner coupling matrix with P_F as weights
t0 = time.time()
couple_pk  = Wigner3j.CoupleMat(lambda_max, pk_L)
coupling_pk = couple_pk.compute_matrix()
print(f'Coupling matrix computed in {time.time()-t0:.1f}s')

# Theory (original formula)
C_theory      = coupling_pk @ PLKjKk / (4*np.pi) / (2*np.pi * chi_bar**2)
C_theory_plot = C_theory / (4*np.pi)**2

## 3. Binning & Comparison

In [ ]:
MD   = MaskDeconvolution(Nl, wl_ref)
bins = MD.binning_matrix('linear', 0, NperBin)
ells = np.arange(Nl, dtype=float)
binned_ells = bins @ ells

binned_theory   = bins @ C_theory_plot[:Nl]
binned_mean     = bins @ cl_mean
binned_std      = bins @ (cl_std / np.sqrt(num_sim))

print(f'{"ell":>8s} {"theory":>12s} {"measured":>12s} {"ratio":>10s}')
print('-' * 46)

ratios_raw = []
for i in range(min(15, len(binned_ells))):
    if binned_theory[i] > 0:
        r_raw = binned_mean[i] / binned_theory[i]
        ratios_raw.append(r_raw)
        print(f'{binned_ells[i]:8.1f} {binned_theory[i]:12.4e} '
              f'{binned_mean[i]:12.4e} {r_raw:10.4f}')

mean_ratio_raw_low = np.mean([r for r, e in zip(ratios_raw[1:9], binned_ells[1:9])])
mean_ratio_raw_all = np.mean(ratios_raw[1:])
print(f'\nMean ratio (ell < 288) = {mean_ratio_raw_low:.4f}')
print(f'Mean ratio (all bins)  = {mean_ratio_raw_all:.4f}')

## 4. The Money Plot — Measured vs. Theory

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: absolute comparison ---
ax = axes[0]
ax.plot(binned_ells, binned_theory, 'k.--', lw=2,
        label=r'Theory: $M_{\ell L}\,C_L^{\rm true}$')
for i in range(num_sim):
    ax.plot(binned_ells, bins @ cl_stack[i], 'C0-', alpha=0.15)
ax.errorbar(binned_ells, binned_mean, yerr=binned_std,
            fmt='C3o', ms=5, capsize=3, label=f'Measured mean ({num_sim} sims)')
ax.set_xlabel(r'multipole $\ell$')
ax.set_ylabel(r'$C_\ell(k{=}0)$')
ax.set_title('Pseudo-$C_\\ell$ (pair-counting theory)')
ax.legend(fontsize=10)

# --- Right: ratio ---
ax = axes[1]
ax.axhline(1.0, color='k', ls='--', lw=1)
ax.errorbar(binned_ells, np.array(ratios_raw),
            yerr=binned_std[:len(ratios_raw)] / binned_theory[:len(ratios_raw)],
            fmt='C3o', ms=5, capsize=3, label='Measured / theory')
ax.set_xlabel(r'multipole $\ell$')
ax.set_ylabel(r'measured / theory')
ax.set_ylim(0.5, 2.0)
ax.set_title(f'Ratio (low-$\\ell$ mean = {mean_ratio_raw_low:.3f})')
ax.legend()

plt.tight_layout()
plt.savefig('plots/money_plot_k0.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('Saved plots/money_plot_k0.pdf')

## 5. MaskDeconvolution Approach

Using $C_\ell^{\rm true} = P_F(\ell/\bar\chi, k)\;/\;(32\pi^3 \bar\chi^2)$ with the `MaskDeconvolution` forward model.

Both the measurement and theory are binned + deconvolved via `MD(pseudo_Cl, bins)` and `MD.convolve_theory_Cls(C_true, bins)`.

In [ ]:
# C_true for MaskDeconvolution: P_F(l/chi_bar, k) / (32 pi^3 chi_bar^2)
# The 32 pi^3 = 2pi x 4pi x 4pi comes from:
#   - 2pi: sFB Limber projection convention (radial FT normalization)
#   - 4pi: addition theorem relating pair-counting to angular Cl
#   - 4pi: (2L+1)/(4pi) prefactor in mode-coupling matrix Mll
cl_true_md = b1_ref**2 * plin_ref(ells / chi_bar) / (32 * np.pi**3 * chi_bar**2)

# Convolve theory and deconvolve measurement (raw — no SN subtraction)
ells_th, theory_dec = MD.convolve_theory_Cls(cl_true_md, bins)
ells_meas, meas_dec = MD(cl_mean, bins)

# Per-realisation scatter for error bars
meas_dec_stack = []
for i in range(num_sim):
    _, md_i = MD(cl_stack[i], bins)
    meas_dec_stack.append(md_i)
meas_dec_stack = np.array(meas_dec_stack)
meas_dec_std = np.std(meas_dec_stack, axis=0) / np.sqrt(num_sim)

print(f'{"ell":>8s} {"theory_dec":>12s} {"meas_dec":>12s} {"ratio":>8s}')
print('-' * 44)
ratios_md = []
for i in range(min(15, len(ells_th))):
    if theory_dec[i] > 0:
        r = meas_dec[i] / theory_dec[i]
        ratios_md.append(r)
        print(f'{ells_th[i]:8.1f} {theory_dec[i]:12.4e} '
              f'{meas_dec[i]:12.4e} {r:8.4f}')

print(f'\nDeconvolved ratio (excl. monopole) = {np.mean(ratios_md[1:]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: absolute, deconvolved ---
ax = axes[0]
ax.plot(ells_th, theory_dec, 'k.--', lw=2, label='Theory (deconvolved)')
ax.errorbar(ells_meas, meas_dec, yerr=meas_dec_std,
            fmt='C1o', ms=5, capsize=3,
            label=f'Deconvolved ({num_sim} sims)')
ax.set_xlabel(r'multipole $\ell$')
ax.set_ylabel(r'$\hat{C}_b(k{=}0)$')
ax.set_title('MaskDeconvolution')
ax.legend(fontsize=10)

# --- Right: ratio comparison ---
ax = axes[1]
ax.axhline(1.0, color='k', ls='--', lw=1)
ax.errorbar(ells_th[:len(ratios_md)], ratios_md,
            yerr=meas_dec_std[:len(ratios_md)] / theory_dec[:len(ratios_md)],
            fmt='C1o', ms=5, capsize=3, label='Deconvolved')
ax.plot(binned_ells[:len(ratios_raw)], np.array(ratios_raw),
        'C3s', ms=5, alpha=0.7, label='Pair-counting')
ax.set_xlabel(r'multipole $\ell$')
ax.set_ylabel('measured / theory')
ax.set_ylim(0.5, 2.0)
ax.set_title(f'Ratio (deconv mean = {np.mean(ratios_md[1:]):.3f})')
ax.legend()

plt.tight_layout()
plt.savefig('plots/money_plot_deconv_k0.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('Saved plots/money_plot_deconv_k0.pdf')

## 6. Angular Window Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(ells, wl_ref, label=r'$W_\ell$ (from $N \times \mathbf{1}$)')
sn = (N * Nskew_0)**2 / (4 * np.pi)
ax.axhline(sn, color='r', ls=':', label=f'Shot noise = $(N \cdot N_{{skew}})^2 / 4\pi$')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$W_\ell$')
ax.set_title('Angular window power spectrum')
ax.legend()

ax = axes[1]
# Mode coupling matrix diagonal
Mll = MD.Mll
ax.semilogy(ells, np.diag(Mll), label=r'$M_{\ell\ell}$')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$M_{\ell\ell}$')
ax.set_title('Mode-coupling matrix diagonal')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
print('=== Pipeline Summary ===')
print(f'Multipoles: ell = 0..{Nl-1}')
print(f'Sightlines: {Nskew_0}')
print(f'Box: N={N}, L={L_box:.1f} Mpc/h, dchi={dchi:.4f} Mpc/h')
print(f'chi_bar = {chi_bar:.1f} Mpc/h')
print(f'Bias: b1 = {b1_ref}, add_rsd = {add_rsd_}')
print(f'Simulations: {num_sim}')
print(f'lambda_max: {lambda_max}')
print(f'\nPair-counting ratio:')
print(f'  low-ell mean: {mean_ratio_raw_low:.4f}')
print(f'  all bins:     {mean_ratio_raw_all:.4f}')
print(f'\nMaskDeconvolution ratio:')
print(f'  excl. monopole: {np.mean(ratios_md[1:]):.4f}')
print(f'\nTheory: C_true = b1^2 * P_lin(ell/chi_bar) / (32 pi^3 chi_bar^2)')
print(f'  32 pi^3 = {32*np.pi**3:.2f} = 2pi x 4pi x 4pi')
print(f'\nNote: No shot-noise subtraction — Sigma w_j^2 / (4 pi) is the')
print(f'j=k diagonal in the pair sum, which is cosmological signal.')